In [16]:
import os
os.makedirs("../data/document_processing_assistant/", exist_ok=True)

file_info = {
    "../data/document_processing_assistant/document.txt":
    """
To refresh your OAuth token, send a POST request to `/auth/token` with grant_type, refresh_token, client_id and client_secret. For CI/CD setup, ensure the .env file is populated with valid API keys. Build steps should be defined in `.github/workflows` dir. New engineers should complete the onboarding checklist within their first week including dev environment setup.

    """
}

for file_path, content in file_info.items():
    with open(file_path, 'w', encoding='utf-8') as file:
        file.write(content)

In [17]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
class DocumentLoader:
    def __init__(self, file_path: str) -> None:
        self.file_path = file_path

    def load_document(self) -> list[str]:
        paragraphs = []
        file = None
        try:
            file = open(self.file_path, 'r', encoding='utf-8')
            content = file.read()
            paragraphs = content.split("\n\n")
            file.close()
        except FileNotFoundError as e:
            print(f"File not found: ", e)
        return paragraphs

    def processes_documents(self, docs: list[str]) -> list[str]:
        processed_documents = [doc.lower() for doc in docs]
        return processed_documents

    def document_chunking(self, docs: list[str], chunk_size: int = 100, chunk_overlap: int = 20) -> list[Document]:
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["", " ", "\n", "\n\n"]
        )
        documents = [
            Document(
                page_content=doc,
            )
            for doc in docs
            ]
        chunks = text_splitter.split_documents(documents)
        return chunks

In [18]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

class VectorStoreBuilder:
    def __init__(self):
        self.embedding_model = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2"
        )
        self.vector_store = None

    def build_vectorstore(self, docs: list[str]):
        documents = [
            Document(
                page_content=doc
            )
            for doc in docs
        ]
        self.vector_store = FAISS.from_documents(documents, self.embedding_model)

    def query(self, query: str, k: int = 3) -> list[Document]:
        if self.vector_store is None:
            return []
        results = self.vector_store.similarity_search(
            query,
            k
        )
        return results

In [21]:
document_loader = DocumentLoader("../data/document_processing_assistant/110123099_sanchitRathore.pdf")
vector_store = VectorStoreBuilder()

paragraphs = document_loader.load_document()
processed_paragraphs = document_loader.processes_documents(paragraphs)

vector_store.build_vectorstore(processed_paragraphs)

result = vector_store.query("Oauth token refresh", 5)
print(result[0].page_content)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2269.19it/s]


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xb5 in position 11: invalid start byte